# 🏢 TechCorp Brasil — Predição de Attrition de Funcionários
**Pós-Graduação em Engenharia de Dados | Disciplina: Data Science Avançado**

---

In [0]:
# Instalação e atualização de dependências no escopo do notebook
%pip install --upgrade numpy pandas scipy scikit-learn imbalanced-learn lightgbm xgboost --quiet

## 1. Configuração do Ambiente e Imports

In [0]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu, pointbiserialr

from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
)
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    precision_recall_curve, roc_curve, auc, fbeta_score,
    matthews_corrcoef, balanced_accuracy_score, ConfusionMatrixDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder, StandardScaler, PolynomialFeatures, OrdinalEncoder
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance

from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.pipeline import Pipeline as ImbPipeline

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

# Paleta padronizada para o trabalho
PALETTE_ATTRITION = {"No": "#2ECC71", "Yes": "#E74C3C"}
sns.set_theme(style="whitegrid", context="notebook", palette="muted")
plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 13, "axes.labelsize": 11})

print("✅ Ambiente configurado com sucesso.")


## 2. Geração do Dataset Sintético

O dataset é gerado sinteticamente inspirado no **IBM HR Analytics**, com 1 milhão de registros e 35 variáveis.
A variável-alvo `Attrition` é criada via função logística com coeficientes interpretáveis (sem data leakage).


In [0]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def clip_int(x, low, high):
    return np.clip(np.round(x), low, high).astype(int)

def generate_ibm_hr_synthetic(n=1470, seed=42):
    """
    Gera base sintética inspirada no IBM HR Analytics.
    Attrition é gerada via função logística — AttritionProbability_hidden
    NÃO é exportada para evitar data leakage perfeito.
    """
    rng = np.random.default_rng(seed)
    employee_number = np.arange(1, n + 1)
    age = clip_int(rng.normal(37, 9, n), 18, 60)
    gender = rng.choice(["Male", "Female"], size=n, p=[0.60, 0.40])

    marital_status = []
    for a in age:
        if a < 28:
            marital_status.append(rng.choice(["Single","Married","Divorced"], p=[0.58,0.37,0.05]))
        elif a < 42:
            marital_status.append(rng.choice(["Single","Married","Divorced"], p=[0.23,0.65,0.12]))
        else:
            marital_status.append(rng.choice(["Single","Married","Divorced"], p=[0.11,0.69,0.20]))
    marital_status = np.array(marital_status)

    department = rng.choice(
        ["Research & Development","Sales","Human Resources"],
        size=n, p=[0.65, 0.30, 0.05]
    )
    education = rng.choice([1,2,3,4,5], size=n, p=[0.12,0.20,0.39,0.24,0.05])

    education_field = []
    for d in department:
        if d == "Research & Development":
            education_field.append(rng.choice(["Life Sciences","Medical","Technical Degree","Other"], p=[0.38,0.35,0.20,0.07]))
        elif d == "Sales":
            education_field.append(rng.choice(["Marketing","Life Sciences","Other","Technical Degree"], p=[0.52,0.20,0.20,0.08]))
        else:
            education_field.append(rng.choice(["Human Resources","Other","Life Sciences"], p=[0.65,0.25,0.10]))
    education_field = np.array(education_field)

    job_role = []
    for d in department:
        if d == "Research & Development":
            job_role.append(rng.choice(
                ["Research Scientist","Laboratory Technician","Manufacturing Director",
                 "Healthcare Representative","Research Director","Manager"],
                p=[0.34,0.32,0.12,0.12,0.04,0.06]
            ))
        elif d == "Sales":
            job_role.append(rng.choice(["Sales Executive","Sales Representative","Manager"], p=[0.70,0.22,0.08]))
        else:
            job_role.append(rng.choice(["Human Resources","Manager"], p=[0.82,0.18]))
    job_role = np.array(job_role)

    total_working_years = clip_int((age - 18) * rng.uniform(0.45, 0.95, n) + rng.normal(0, 2.2, n), 0, 40)
    job_level = clip_int(1 + (total_working_years / 9) + rng.normal(0, 0.7, n), 1, 5)

    years_at_company = np.array([rng.integers(0, max(1, twy + 1)) if twy > 0 else 0 for twy in total_working_years])
    years_at_company = np.minimum(years_at_company, total_working_years)
    years_in_current_role = np.array([rng.integers(0, yac + 1) if yac > 0 else 0 for yac in years_at_company])
    years_with_curr_manager = np.array([rng.integers(0, yac + 1) if yac > 0 else 0 for yac in years_at_company])
    years_since_last_promotion = np.array([rng.integers(0, min(yac + 1, 16)) if yac > 0 else 0 for yac in years_at_company])
    num_companies_worked = clip_int(rng.poisson(lam=np.maximum(1.2, total_working_years / 8)), 0, 9)
    distance_from_home = clip_int(rng.gamma(shape=2.0, scale=4.8, size=n), 1, 29)
    business_travel = rng.choice(["Non-Travel","Travel_Rarely","Travel_Frequently"], size=n, p=[0.10, 0.70, 0.20])

    environment_satisfaction = rng.choice([1,2,3,4], size=n, p=[0.18,0.19,0.31,0.32])
    job_satisfaction = rng.choice([1,2,3,4], size=n, p=[0.17,0.20,0.32,0.31])
    relationship_satisfaction = rng.choice([1,2,3,4], size=n, p=[0.16,0.21,0.32,0.31])
    job_involvement = rng.choice([1,2,3,4], size=n, p=[0.08,0.25,0.50,0.17])
    work_life_balance = rng.choice([1,2,3,4], size=n, p=[0.08,0.24,0.45,0.23])

    performance_rating = rng.choice([3,4], size=n, p=[0.85, 0.15])
    percent_salary_hike = clip_int(rng.normal(13 + 3 * (performance_rating == 4), 3, n), 11, 25)

    stock_option_level = []
    for m in marital_status:
        if m == "Single":
            stock_option_level.append(rng.choice([0,1,2,3], p=[0.58,0.28,0.10,0.04]))
        else:
            stock_option_level.append(rng.choice([0,1,2,3], p=[0.34,0.41,0.18,0.07]))
    stock_option_level = np.array(stock_option_level)

    over_time = []
    for wlb, ji in zip(work_life_balance, job_involvement):
        p_ot = 0.18 + 0.08 * (wlb <= 2) + 0.05 * (ji >= 3)
        over_time.append(rng.choice(["No","Yes"], p=[1 - p_ot, p_ot]))
    over_time = np.array(over_time)

    training_times_last_year = clip_int(rng.poisson(2.3, n), 0, 6)
    role_bonus_map = {
        "Manager": 3000, "Research Director": 3500, "Manufacturing Director": 1300,
        "Healthcare Representative": 950, "Sales Executive": 850,
        "Research Scientist": 450, "Laboratory Technician": -250,
        "Sales Representative": -450, "Human Resources": 100
    }
    role_bonus = np.array([role_bonus_map[r] for r in job_role])
    base_income = (1300 + job_level * 2400 + total_working_years * 105
                   + (performance_rating == 4) * 600 + rng.normal(0, 850, n))
    monthly_income = clip_int(base_income + role_bonus, 1000, 22000)
    monthly_rate = clip_int(monthly_income * rng.uniform(1.2, 1.9, n) + rng.normal(0, 900, n), 2000, 27000)
    daily_rate = clip_int(rng.normal(800, 400, n), 100, 1500)
    hourly_rate = clip_int(rng.normal(66, 20, n), 30, 100)

    logit = (
        -1.85
        + 0.82  * (over_time == "Yes")
        + 0.52  * (business_travel == "Travel_Frequently")
        + 0.20  * (business_travel == "Travel_Rarely")
        + 0.42  * (marital_status == "Single")
        - 0.20  * (marital_status == "Married")
        + 0.024 * distance_from_home
        - 0.035 * (age - 35)
        - 0.000065 * (monthly_income - 6000)
        - 0.23  * (environment_satisfaction - 2.5)
        - 0.27  * (job_satisfaction - 2.5)
        - 0.18  * (work_life_balance - 2.5)
        - 0.19  * (job_involvement - 2.5)
        - 0.07  * years_at_company
        + 0.12  * num_companies_worked
        - 0.18  * stock_option_level
        + 0.28  * (job_role == "Sales Representative")
        + 0.18  * (job_role == "Laboratory Technician")
        - 0.10  * training_times_last_year
    )
    attrition_probability = sigmoid(logit)
    attrition = rng.binomial(1, attrition_probability)

    df = pd.DataFrame({
        "Age": age, "Attrition": np.where(attrition == 1, "Yes", "No"),
        "BusinessTravel": business_travel, "DailyRate": daily_rate,
        "Department": department, "DistanceFromHome": distance_from_home,
        "Education": education, "EducationField": education_field,
        "EmployeeCount": np.ones(n, dtype=int), "EmployeeNumber": employee_number,
        "EnvironmentSatisfaction": environment_satisfaction, "Gender": gender,
        "HourlyRate": hourly_rate, "JobInvolvement": job_involvement,
        "JobLevel": job_level, "JobRole": job_role, "JobSatisfaction": job_satisfaction,
        "MaritalStatus": marital_status, "MonthlyIncome": monthly_income,
        "MonthlyRate": monthly_rate, "NumCompaniesWorked": num_companies_worked,
        "Over18": np.repeat("Y", n), "OverTime": over_time,
        "PercentSalaryHike": percent_salary_hike, "PerformanceRating": performance_rating,
        "RelationshipSatisfaction": relationship_satisfaction,
        "StandardHours": np.repeat(80, n), "StockOptionLevel": stock_option_level,
        "TotalWorkingYears": total_working_years, "TrainingTimesLastYear": training_times_last_year,
        "WorkLifeBalance": work_life_balance, "YearsAtCompany": years_at_company,
        "YearsInCurrentRole": years_in_current_role,
        "YearsSinceLastPromotion": years_since_last_promotion,
        "YearsWithCurrManager": years_with_curr_manager,
    })
    return df

df = generate_ibm_hr_synthetic(n=1_000_000, seed=42)
print(f"Dataset gerado: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
df.head()


## 3. Limpeza e Pré-processamento Inicial

In [0]:
# Verificação de nulos
nulos = df.isnull().sum()
colunas_com_nulos = nulos[nulos > 0]
if colunas_com_nulos.empty:
    print("✅ Nenhum valor nulo encontrado.")
else:
    display(colunas_com_nulos.to_frame(name="Total de Nulos"))

# Removendo colunas constantes e identificadores
colunas_para_remover = ["EmployeeCount", "StandardHours", "Over18", "EmployeeNumber"]
df_clean = df.drop(columns=[c for c in colunas_para_remover if c in df.columns])

# Target numérico
df_clean["Attrition_Target"] = (df_clean["Attrition"] == "Yes").astype(int)

print(f"Shape após limpeza: {df_clean.shape}")
print(f"Taxa de Attrition (desbalanceamento): {df_clean['Attrition_Target'].mean():.2%}")
print(f"  - Ficam (No):  {(df_clean['Attrition']=='No').sum():,}")
print(f"  - Saem (Yes):  {(df_clean['Attrition']=='Yes').sum():,}")


## 4. Análise Exploratória de Dados (EDA)

A EDA é dividida em blocos temáticos para facilitar a narrativa analítica:
1. Distribuição do target e desbalanceamento
2. Variáveis numéricas (distribuições e boxplots por attrition)
3. Variáveis categóricas (taxas de attrition por categoria)
4. Correlações e informação mútua
5. Análise demográfica (gênero, estado civil, idade)
6. Análise de satisfação e engajamento
7. Análise financeira e de carreira


In [0]:
# ── 4.1  Distribuição do Target e Desbalanceamento ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Countplot
ax0 = sns.countplot(data=df_clean, x="Attrition",
                    palette=PALETTE_ATTRITION, order=["No","Yes"], ax=axes[0])
axes[0].set_title("Distribuição de Attrition (Absoluta)", fontweight="bold")
axes[0].set_xlabel("")
for p in ax0.patches:
    ax0.annotate(f"{int(p.get_height()):,}\n({p.get_height()/len(df_clean):.1%})",
                 (p.get_x() + p.get_width()/2, p.get_height()),
                 ha="center", va="bottom", fontsize=10)

# Pizza
sizes = df_clean["Attrition"].value_counts()
axes[1].pie(sizes, labels=sizes.index, colors=[PALETTE_ATTRITION[k] for k in sizes.index],
            autopct="%1.1f%%", startangle=90, textprops={"fontsize": 12})
axes[1].set_title("Proporção de Attrition", fontweight="bold")

plt.suptitle("Desbalanceamento: ~16% de Attrition", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print("\n⚠️  Dataset altamente desbalanceado (~16% positivos). Métricas como PR-AUC e F2-Score serão priorizadas sobre Accuracy.")


In [0]:
# ── 4.2  Distribuições de Variáveis Numéricas por Attrition ─────────────────
num_vars = ["Age", "MonthlyIncome", "TotalWorkingYears", "YearsAtCompany",
            "DistanceFromHome", "NumCompaniesWorked", "YearsSinceLastPromotion",
            "TrainingTimesLastYear"]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, var in enumerate(num_vars):
    sns.kdeplot(data=df_clean, x=var, hue="Attrition",
                palette=PALETTE_ATTRITION, fill=True, alpha=0.4, ax=axes[i])
    axes[i].set_title(f"Distribuição: {var}", fontweight="bold")
    axes[i].set_xlabel("")

plt.suptitle("Distribuição de Variáveis Numéricas por Attrition (KDE)", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()


In [0]:
# ── 4.3  Boxplots: Variáveis Numéricas vs Attrition ─────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, var in enumerate(num_vars):
    sns.boxplot(data=df_clean, x="Attrition", y=var,
                palette=PALETTE_ATTRITION, order=["No","Yes"], ax=axes[i],
                showfliers=False)
    # Adiciona médias
    medias = df_clean.groupby("Attrition")[var].median()
    for j, (label, med) in enumerate(medias.items()):
        axes[i].text(j, med, f'{med:.1f}', ha='center', va='bottom',
                     fontsize=9, color='black', fontweight='bold')
    axes[i].set_title(f"{var}", fontweight="bold")
    axes[i].set_xlabel("")

plt.suptitle("Boxplots: Variáveis Numéricas vs Attrition (mediana anotada)", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()


In [0]:
# ── 4.4  Taxa de Attrition por Variáveis Categóricas ────────────────────────
cat_vars = ["Department", "JobRole", "MaritalStatus", "BusinessTravel",
            "OverTime", "Gender", "EducationField", "JobLevel"]

fig, axes = plt.subplots(2, 4, figsize=(22, 12))
axes = axes.flatten()

for i, var in enumerate(cat_vars):
    taxa = df_clean.groupby(var)["Attrition_Target"].mean().sort_values(ascending=False) * 100
    bars = sns.barplot(x=taxa.values, y=taxa.index, palette="RdYlGn_r", ax=axes[i])
    axes[i].set_title(f"Attrition (%) por {var}", fontweight="bold")
    axes[i].set_xlabel("% Attrition")
    axes[i].axvline(df_clean["Attrition_Target"].mean()*100, color="navy",
                    linestyle="--", linewidth=1.2, label="Média geral")
    for p in bars.patches:
        axes[i].text(p.get_width() + 0.3, p.get_y() + p.get_height()/2,
                     f'{p.get_width():.1f}%', va='center', fontsize=8)

plt.suptitle("Taxa de Attrition por Variáveis Categóricas", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()
print("\n📌 Insight: OverTime='Yes', estado civil 'Single' e 'Sales Representative' apresentam taxas de attrition muito acima da média.")


In [0]:
# ── 4.5  Heatmap de Correlação (Variáveis Numéricas) ────────────────────────
num_df = df_clean.select_dtypes(include=[np.number]).drop(columns=["Attrition_Target"])
corr = num_df.corr()

# Máscara triangular superior
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(corr, mask=mask, annot=False, cmap="coolwarm", center=0,
            linewidths=0.4, ax=ax, vmin=-1, vmax=1)
ax.set_title("Heatmap de Correlação — Variáveis Numéricas", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [0]:
# ── 4.6  Correlação com o Target (Ponto-biserial) ───────────────────────────
corr_target = {}
for col in df_clean.select_dtypes(include=[np.number]).columns:
    if col != "Attrition_Target":
        r, p = pointbiserialr(df_clean["Attrition_Target"], df_clean[col])
        corr_target[col] = {"Correlação": round(r, 4), "p-value": round(p, 6)}

df_corr = pd.DataFrame(corr_target).T.sort_values("Correlação", key=abs, ascending=False)
print("Correlação Ponto-Biserial com Attrition_Target:")
display(df_corr.head(15))

fig, ax = plt.subplots(figsize=(10, 8))
cores = ["#E74C3C" if v > 0 else "#2ECC71" for v in df_corr["Correlação"]]
df_corr["Correlação"].plot(kind="barh", color=cores, ax=ax)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Correlação Ponto-Biserial com Attrition", fontweight="bold")
ax.set_xlabel("Correlação")
plt.tight_layout()
plt.show()


In [0]:
# ── 4.7  Informação Mútua (captura relações não-lineares) ───────────────────
X_mi = df_clean.drop(columns=["Attrition","Attrition_Target"])
# Encoding simples para MI
X_mi_enc = X_mi.copy()
for col in X_mi_enc.select_dtypes(include="object").columns:
    X_mi_enc[col] = X_mi_enc[col].astype("category").cat.codes

mi_scores = mutual_info_classif(X_mi_enc, df_clean["Attrition_Target"], random_state=42)
mi_series = pd.Series(mi_scores, index=X_mi_enc.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
mi_series.head(15).plot(kind="barh", color="#3498DB", ax=ax)
ax.set_title("Top 15 Features por Informação Mútua com Attrition", fontweight="bold")
ax.set_xlabel("Informação Mútua")
ax.invert_yaxis()
plt.tight_layout()
plt.show()
print("\n📌 Informação mútua captura relações não-lineares ignoradas pela correlação de Pearson.")


In [0]:
# ── 4.8  Análise Demográfica: Attrition por Faixa Etária ────────────────────
df_clean["FaixaEtaria"] = pd.cut(df_clean["Age"], bins=[17,25,35,45,55,60],
                                  labels=["18-25","26-35","36-45","46-55","56-60"])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Taxa por faixa etária
taxa_idade = df_clean.groupby("FaixaEtaria")["Attrition_Target"].mean() * 100
sns.barplot(x=taxa_idade.index.astype(str), y=taxa_idade.values,
            palette="coolwarm", ax=axes[0])
axes[0].set_title("Taxa de Attrition por Faixa Etária", fontweight="bold")
axes[0].set_ylabel("% Attrition")
axes[0].axhline(df_clean["Attrition_Target"].mean()*100, color="navy",
                linestyle="--", label="Média geral")
axes[0].legend()

# Violinplot Idade por Attrition
sns.violinplot(data=df_clean, x="Attrition", y="Age",
               palette=PALETTE_ATTRITION, order=["No","Yes"], ax=axes[1])
axes[1].set_title("Distribuição de Idade por Attrition (Violin)", fontweight="bold")

plt.tight_layout()
plt.show()
print("\n📌 Insight: Funcionários mais jovens (18-25 anos) têm taxa de attrition significativamente maior.")


In [0]:
# ── 4.9  Análise de Satisfação e Engajamento ────────────────────────────────
satisf_vars = ["JobSatisfaction", "EnvironmentSatisfaction",
               "RelationshipSatisfaction", "WorkLifeBalance", "JobInvolvement"]

fig, axes = plt.subplots(1, len(satisf_vars), figsize=(22, 6))

for i, var in enumerate(satisf_vars):
    taxa = df_clean.groupby(var)["Attrition_Target"].mean() * 100
    sns.barplot(x=taxa.index, y=taxa.values, palette="RdYlGn_r", ax=axes[i])
    axes[i].set_title(f"{var}", fontweight="bold", fontsize=10)
    axes[i].set_ylabel("% Attrition" if i == 0 else "")
    axes[i].axhline(df_clean["Attrition_Target"].mean()*100,
                    color="navy", linestyle="--", linewidth=1)

plt.suptitle("Taxa de Attrition por Nível de Satisfação/Engajamento (1=baixo, 4=alto)",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print("\n📌 Insight: Todos os índices de satisfação apresentam relação inversa com attrition — funcionários menos satisfeitos saem mais.")


In [0]:
# ── 4.10  Attrition por Estado Civil e Gênero (heatmap cruzado) ─────────────
pivot = df_clean.pivot_table(values="Attrition_Target",
                              index="MaritalStatus", columns="Gender",
                              aggfunc="mean") * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlGn_r",
            linewidths=0.5, ax=axes[0], vmin=0, vmax=30)
axes[0].set_title("Attrition (%) — Estado Civil × Gênero", fontweight="bold")

# Attrition por viagens de negócios e OverTime combinados
pivot2 = df_clean.pivot_table(values="Attrition_Target",
                               index="BusinessTravel", columns="OverTime",
                               aggfunc="mean") * 100
sns.heatmap(pivot2, annot=True, fmt=".1f", cmap="RdYlGn_r",
            linewidths=0.5, ax=axes[1], vmin=0, vmax=50)
axes[1].set_title("Attrition (%) — Viagem de Negócios × Hora Extra", fontweight="bold")

plt.tight_layout()
plt.show()
print("\n📌 Insight: Combinação de OverTime=Yes + Travel_Frequently representa o cenário de maior risco.")


In [0]:
# ── 4.11  Análise Financeira: Renda Mensal por Cargo e Attrition ─────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Renda por cargo com taxa de attrition sobreposta
cargo_renda = df_clean.groupby("JobRole")["MonthlyIncome"].median().sort_values()
cargo_taxa  = df_clean.groupby("JobRole")["Attrition_Target"].mean() * 100

ax2 = axes[0].twinx()
cargo_renda.plot(kind="barh", color="#3498DB", alpha=0.7, ax=axes[0])
ax2.plot(cargo_taxa[cargo_renda.index].values, range(len(cargo_renda)),
         "ro-", markersize=8, label="% Attrition")
axes[0].set_title("Renda Mediana vs Taxa de Attrition por Cargo", fontweight="bold")
axes[0].set_xlabel("Renda Mediana (R$)")
ax2.set_ylabel("% Attrition")
ax2.legend(loc="lower right")

# Distribuição de renda por nível de cargo
sns.boxplot(data=df_clean, x="JobLevel", y="MonthlyIncome",
            hue="Attrition", palette=PALETTE_ATTRITION, ax=axes[1])
axes[1].set_title("Renda Mensal por Nível de Cargo e Attrition", fontweight="bold")

plt.tight_layout()
plt.show()


In [0]:
# ── 4.12  Testes Estatísticos de Associação ──────────────────────────────────
print("=" * 60)
print("TESTES ESTATÍSTICOS DE ASSOCIAÇÃO COM ATTRITION")
print("=" * 60)

# Qui-Quadrado para variáveis categóricas
cat_test_vars = ["OverTime", "MaritalStatus", "BusinessTravel", "Department",
                 "Gender", "JobRole"]
for var in cat_test_vars:
    ct = pd.crosstab(df_clean[var], df_clean["Attrition_Target"])
    chi2_val, p_val, dof, _ = chi2_contingency(ct)
    sig = "***" if p_val < 0.001 else ("**" if p_val < 0.01 else ("*" if p_val < 0.05 else "ns"))
    print(f"  Chi² ({var:<25}): chi2={chi2_val:>12.1f}  p={p_val:.2e}  {sig}")

print()

# Mann-Whitney para variáveis numéricas
num_test_vars = ["MonthlyIncome", "Age", "YearsAtCompany", "DistanceFromHome",
                 "TotalWorkingYears"]
for var in num_test_vars:
    grupo_0 = df_clean[df_clean["Attrition_Target"]==0][var]
    grupo_1 = df_clean[df_clean["Attrition_Target"]==1][var]
    stat, p_val = mannwhitneyu(grupo_0, grupo_1, alternative="two-sided")
    sig = "***" if p_val < 0.001 else ("**" if p_val < 0.01 else ("*" if p_val < 0.05 else "ns"))
    print(f"  Mann-Whitney ({var:<25}): U={stat:.2e}  p={p_val:.2e}  {sig}")

print("\n  Legenda: *** p<0.001  ** p<0.01  * p<0.05  ns = não significativo")


## 5. Feature Engineering

Criamos **13 features novas** (11 temáticas + 2 polinomiais), cada uma com justificativa técnica e de negócio.
Features calculadas antes do split para evitar look-ahead; a única exceção é `Relative_Income_Dept`
(média por grupo), que em produção viria de um artefato ajustado apenas no treino.


In [0]:
df_featured = df_clean.copy()

# 1. Renda normalizada pelo tempo de empresa — proxy de crescimento salarial
df_featured["Income_Per_Year"] = df_featured["MonthlyIncome"] / (df_featured["YearsAtCompany"] + 1)

# 2. Índice de estagnação de carreira — tempo sem promoção relativo ao total na empresa
df_featured["Stagnation_Index"] = df_featured["YearsSinceLastPromotion"] / (df_featured["YearsAtCompany"] + 1)

# 3. Crescimento salarial relativo à experiência total
df_featured["Income_Experience_Ratio"] = df_featured["MonthlyIncome"] / (df_featured["TotalWorkingYears"] + 1)

# 4. Risco de burnout: hora extra + desequilíbrio trabalho-vida
overtime_bin = (df_featured["OverTime"] == "Yes").astype(int)
df_featured["Burnout_Risk"] = overtime_bin + (5 - df_featured["WorkLifeBalance"])

# 5. Satisfação total composta (soma dos 3 índices)
df_featured["Total_Satisfaction"] = (
    df_featured["EnvironmentSatisfaction"]
    + df_featured["JobSatisfaction"]
    + df_featured["RelationshipSatisfaction"]
)

# 6. Índice de lealdade — anos totais por empresa trabalhada
df_featured["Loyalty_Index"] = df_featured["TotalWorkingYears"] / (df_featured["NumCompaniesWorked"] + 1)

# 7. Renda relativa ao departamento — captura insatisfação por comparação social
dept_avg = df_featured.groupby("Department")["MonthlyIncome"].transform("mean")
df_featured["Relative_Income_Dept"] = df_featured["MonthlyIncome"] / dept_avg

# 8. Friction com gestão — quanto tempo passou sob o mesmo gestor
df_featured["Management_Friction"] = df_featured["YearsWithCurrManager"] / (df_featured["YearsAtCompany"] + 1)

# 9. Estresse de deslocamento — distância ponderada pela frequência de viagens
travel_map = {"Non-Travel": 0, "Travel_Rarely": 1, "Travel_Frequently": 2}
travel_weight = df_featured["BusinessTravel"].map(travel_map)
df_featured["Commute_Stress"] = df_featured["DistanceFromHome"] * (travel_weight + 1)

# 10. Stock options por ano — retenção via equity diluída no tempo
df_featured["Stock_per_Year"] = df_featured["StockOptionLevel"] / (df_featured["YearsAtCompany"] + 1)

# 11. Valor de senioridade — desequilíbrio entre idade e nível do cargo
df_featured["Seniority_Value"] = df_featured["Age"] / df_featured["JobLevel"]

print(f"✅ Feature engineering concluído. Shape: {df_featured.shape}")
print("Novas features criadas:")
new_feats = ["Income_Per_Year","Stagnation_Index","Income_Experience_Ratio",
             "Burnout_Risk","Total_Satisfaction","Loyalty_Index","Relative_Income_Dept",
             "Management_Friction","Commute_Stress","Stock_per_Year","Seniority_Value"]
display(df_featured[new_feats].describe().T.round(3))


In [0]:
# Visualização do impacto das novas features
fig, axes = plt.subplots(3, 4, figsize=(22, 16))
axes = axes.flatten()

new_feats_plot = ["Income_Per_Year","Stagnation_Index","Income_Experience_Ratio",
                  "Burnout_Risk","Total_Satisfaction","Loyalty_Index",
                  "Relative_Income_Dept","Management_Friction","Commute_Stress",
                  "Stock_per_Year","Seniority_Value"]

for i, feat in enumerate(new_feats_plot):
    sns.kdeplot(data=df_featured, x=feat, hue="Attrition",
                palette=PALETTE_ATTRITION, fill=True, alpha=0.4, ax=axes[i])
    axes[i].set_title(f"{feat}", fontweight="bold", fontsize=10)
    axes[i].set_xlabel("")

axes[-1].set_visible(False)
plt.suptitle("Distribuição das Features Criadas por Attrition", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()


## 6. Divisão Treino/Teste e Features Polinomiais

In [0]:
# Target e features
X = df_featured.drop(columns=["Attrition", "Attrition_Target", "FaixaEtaria"])
y = df_featured["Attrition_Target"]

# Split estratificado 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Features polinomiais ajustadas SOMENTE no treino (evita leakage)
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)
poly_cols = ["JobInvolvement", "PerformanceRating"]

poly_train = poly.fit_transform(X_train[poly_cols])   # fit apenas no treino
poly_test  = poly.transform(X_test[poly_cols])         # transforma o teste

X_train = X_train.copy()
X_test  = X_test.copy()

X_train["Involvement_x_Performance"] = poly_train[:, 2]
X_test ["Involvement_x_Performance"] = poly_test [:, 2]
X_train["Involvement_Squared"] = X_train["JobInvolvement"] ** 2
X_test ["Involvement_Squared"] = X_test ["JobInvolvement"] ** 2

print(f"Treino : {X_train.shape[0]:,} registros | Teste: {X_test.shape[0]:,}")
print(f"Attrition no treino: {y_train.mean():.2%} | no teste: {y_test.mean():.2%}")
print(f"Total de features  : {X_train.shape[1]}")


## 7. Pré-processamento e Pipelines

In [0]:
numerical_cols   = X_train.select_dtypes(include=["int64","float64"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object","category"]).columns.tolist()

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(),                                    numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), categorical_cols),
])

# Razão de desbalanceamento para XGBoost
neg = int((y_train == 0).sum())
pos = int((y_train == 1).sum())
scale_pos = neg / pos
print(f"Razão de desbalanceamento (neg/pos): {scale_pos:.2f}")
print(f"  Negativos (ficam): {neg:,}")
print(f"  Positivos (saem) : {pos:,}")


## 8. Modelagem — 4 Algoritmos com Tratamento de Desbalanceamento

Utilizamos `class_weight='balanced'` e `scale_pos_weight` como estratégia primária
de tratamento do desbalanceamento, integrado diretamente nos classificadores.


In [0]:
models_base = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=200, learning_rate=0.03, max_depth=6,
        num_leaves=31, class_weight="balanced", random_state=42, n_jobs=-1,
        verbose=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=6,
        scale_pos_weight=scale_pos, random_state=42, n_jobs=-1, eval_metric="logloss",
        verbosity=0
    ),
}

results = {}

for name, clf in models_base.items():
    print(f"⏳ Treinando: {name}...")

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier",   clf),
    ])

    pipeline.fit(X_train, y_train)
    y_pred  = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    precision, recall, thresholds_pr = precision_recall_curve(y_test, y_proba)
    pr_auc   = auc(recall, precision)
    f2       = fbeta_score(y_test, y_pred, beta=2)
    roc_auc  = roc_auc_score(y_test, y_proba)
    mcc      = matthews_corrcoef(y_test, y_pred)
    bal_acc  = balanced_accuracy_score(y_test, y_pred)

    results[name] = {
        "Pipeline":    pipeline,
        "ROC-AUC":     roc_auc,
        "PR-AUC":      pr_auc,
        "F2-Score":    f2,
        "MCC":         mcc,
        "Bal-Accuracy": bal_acc,
        "Report":      classification_report(y_test, y_pred),
        "y_pred":      y_pred,
        "y_proba":     y_proba,
        "pr_precision": precision,
        "pr_recall":   recall,
        "pr_thresholds": thresholds_pr,
    }
    print(f"  ✅ {name}: PR-AUC={pr_auc:.4f}  ROC-AUC={roc_auc:.4f}  F2={f2:.4f}  MCC={mcc:.4f}")

print("\n✅ Treinamento dos modelos base concluído!")


## 9. Tratamento de Desbalanceamento via SMOTE + ADASYN

Além do `class_weight`, aplicamos **SMOTE** (Synthetic Minority Oversampling Technique)
e **ADASYN** (Adaptive Synthetic Sampling) integrados via `imblearn.Pipeline`,
que garante que o oversampling ocorra SOMENTE no treino (sem contaminar o teste).


In [0]:
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.pipeline import Pipeline as ImbPipeline

# Precisamos de matriz numérica para SMOTE — transformar antes
X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc  = preprocessor.transform(X_test)

smote_results = {}

for sampler_name, sampler in [("SMOTE", SMOTE(random_state=42)),
                               ("ADASYN", ADASYN(random_state=42))]:
    print(f"⏳ Aplicando {sampler_name} + LightGBM...")
    X_res, y_res = sampler.fit_resample(X_train_enc, y_train)
    print(f"  Shape após {sampler_name}: {X_res.shape} | Positivos: {y_res.sum():,}")

    clf = LGBMClassifier(n_estimators=200, learning_rate=0.03, max_depth=6,
                          num_leaves=31, random_state=42, n_jobs=-1, verbose=-1)
    clf.fit(X_res, y_res)
    y_pred  = clf.predict(X_test_enc)
    y_proba = clf.predict_proba(X_test_enc)[:, 1]

    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    pr_auc  = auc(recall, precision)
    f2      = fbeta_score(y_test, y_pred, beta=2)
    roc_auc = roc_auc_score(y_test, y_proba)
    mcc     = matthews_corrcoef(y_test, y_pred)

    smote_results[f"LightGBM + {sampler_name}"] = {
        "PR-AUC": pr_auc, "ROC-AUC": roc_auc, "F2-Score": f2, "MCC": mcc,
        "y_pred": y_pred, "y_proba": y_proba,
        "pr_precision": precision, "pr_recall": recall,
        "Bal-Accuracy": balanced_accuracy_score(y_test, y_pred),
        "Report": classification_report(y_test, y_pred),
    }
    print(f"  ✅ PR-AUC={pr_auc:.4f}  ROC-AUC={roc_auc:.4f}  F2={f2:.4f}")

results.update(smote_results)
print("\n✅ SMOTE e ADASYN concluídos!")


## 10. Otimização de Hiperparâmetros — RandomizedSearchCV

Otimizamos o LightGBM (melhor modelo esperado) via `RandomizedSearchCV` com validação
cruzada estratificada, priorizando **PR-AUC** como métrica de seleção.


In [0]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "classifier__n_estimators":  [100, 200, 300, 500],
    "classifier__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "classifier__max_depth":     [4, 6, 8, -1],
    "classifier__num_leaves":    [20, 31, 50, 70],
    "classifier__min_child_samples": [20, 50, 100],
    "classifier__subsample":     [0.7, 0.8, 0.9, 1.0],
    "classifier__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
}

lgbm_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LGBMClassifier(class_weight="balanced", random_state=42,
                                   n_jobs=-1, verbose=-1)),
])

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    lgbm_pipeline,
    param_distributions=param_dist,
    n_iter=20,          # reduzido para viabilidade computacional (aumentar para produção)
    scoring="average_precision",  # PR-AUC via average_precision
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

print("⏳ Iniciando RandomizedSearchCV (20 iterações × 3-fold CV)...")
search.fit(X_train, y_train)

best_lgbm = search.best_estimator_
y_pred_opt  = best_lgbm.predict(X_test)
y_proba_opt = best_lgbm.predict_proba(X_test)[:, 1]

precision_opt, recall_opt, thresh_opt = precision_recall_curve(y_test, y_proba_opt)
pr_auc_opt   = auc(recall_opt, precision_opt)
f2_opt       = fbeta_score(y_test, y_pred_opt, beta=2)
roc_auc_opt  = roc_auc_score(y_test, y_proba_opt)
mcc_opt      = matthews_corrcoef(y_test, y_pred_opt)

results["LightGBM (Otimizado)"] = {
    "Pipeline": best_lgbm, "PR-AUC": pr_auc_opt, "ROC-AUC": roc_auc_opt,
    "F2-Score": f2_opt, "MCC": mcc_opt,
    "Bal-Accuracy": balanced_accuracy_score(y_test, y_pred_opt),
    "Report": classification_report(y_test, y_pred_opt),
    "y_pred": y_pred_opt, "y_proba": y_proba_opt,
    "pr_precision": precision_opt, "pr_recall": recall_opt,
    "pr_thresholds": thresh_opt,
}

print(f"\n✅ Melhores parâmetros: {search.best_params_}")
print(f"   PR-AUC (otimizado): {pr_auc_opt:.4f}")
print(f"   ROC-AUC           : {roc_auc_opt:.4f}")
print(f"   F2-Score          : {f2_opt:.4f}")
print(f"   MCC               : {mcc_opt:.4f}")


## 11. Comparação Completa de Todos os Modelos

In [0]:
# Tabela comparativa
rows = []
for name, m in results.items():
    rows.append({
        "Modelo":        name,
        "PR-AUC":        round(m.get("PR-AUC",  0), 4),
        "ROC-AUC":       round(m.get("ROC-AUC", 0), 4),
        "F2-Score":      round(m.get("F2-Score",0), 4),
        "MCC":           round(m.get("MCC",     0), 4),
        "Bal-Accuracy":  round(m.get("Bal-Accuracy", 0), 4),
    })

df_metrics = pd.DataFrame(rows).sort_values("PR-AUC", ascending=False)
print("COMPARAÇÃO DE MÉTRICAS — TODOS OS MODELOS")
print("=" * 75)
display(df_metrics)

# Identificando melhor modelo
best_model_name = df_metrics.iloc[0]["Modelo"]
print(f"\n🏆 Melhor modelo por PR-AUC: {best_model_name}")


In [0]:
# Curvas Precision-Recall — todos os modelos
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

colors_map = ["#E74C3C","#3498DB","#2ECC71","#F39C12","#9B59B6","#1ABC9C","#E67E22"]
baseline = y_test.mean()

# PR Curves
for (name, m), color in zip(results.items(), colors_map):
    if "pr_precision" in m:
        axes[0].plot(m["pr_recall"], m["pr_precision"],
                     label=f"{name} ({m['PR-AUC']:.3f})", linewidth=2, color=color)

axes[0].axhline(baseline, linestyle="--", color="gray",
                label=f"Baseline ({baseline:.2%})")
axes[0].set_xlabel("Recall", fontsize=12)
axes[0].set_ylabel("Precision", fontsize=12)
axes[0].set_title("Curvas Precision-Recall", fontsize=13, fontweight="bold")
axes[0].legend(fontsize=8, loc="upper right")

# ROC Curves
for (name, m), color in zip(results.items(), colors_map):
    if "y_proba" in m:
        fpr, tpr, _ = roc_curve(y_test, m["y_proba"])
        axes[1].plot(fpr, tpr, label=f"{name} ({m['ROC-AUC']:.3f})",
                     linewidth=2, color=color)

axes[1].plot([0,1],[0,1], "k--", label="Aleatório")
axes[1].set_xlabel("FPR (1 - Especificidade)", fontsize=12)
axes[1].set_ylabel("TPR (Sensibilidade)", fontsize=12)
axes[1].set_title("Curvas ROC", fontsize=13, fontweight="bold")
axes[1].legend(fontsize=8)

plt.suptitle("Curvas de Avaliação — Comparação dos Modelos", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()


## 12. Otimização de Threshold

O threshold padrão (0.5) não é ótimo para dados desbalanceados.
Buscamos o ponto que maximiza o **F2-Score** (que prioriza recall, reduzindo falsos negativos —
cada funcionário de risco não identificado representa custo de R$ 1,5× salário anual).


In [0]:
best_pipeline_obj = results[best_model_name]
y_proba_best = best_pipeline_obj["y_proba"]

# Grid de thresholds
thresholds = np.linspace(0.05, 0.95, 200)
f2_scores, precision_scores, recall_scores = [], [], []

for t in thresholds:
    y_pred_t = (y_proba_best >= t).astype(int)
    f2_scores.append(fbeta_score(y_test, y_pred_t, beta=2, zero_division=0))
    p, r = precision_recall_curve(y_test, y_proba_best)[:2]
    # Precision e recall no threshold exato
    prec = (y_test[y_pred_t == 1]).mean() if y_pred_t.sum() > 0 else 0
    rec  = (y_pred_t[y_test == 1]).mean()
    precision_scores.append(prec)
    recall_scores.append(rec)

best_thresh_idx = np.argmax(f2_scores)
best_threshold  = thresholds[best_thresh_idx]
best_f2_thresh  = f2_scores[best_thresh_idx]

print(f"🎯 Threshold ótimo (max F2-Score): {best_threshold:.3f}")
print(f"   F2-Score no threshold ótimo : {best_f2_thresh:.4f}")
print(f"   F2-Score com threshold=0.5  : {fbeta_score(y_test, (y_proba_best>=0.5).astype(int), beta=2):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# F2-Score por threshold
axes[0].plot(thresholds, f2_scores, color="#E74C3C", linewidth=2)
axes[0].axvline(best_threshold, color="navy", linestyle="--",
                label=f"Threshold ótimo: {best_threshold:.3f}")
axes[0].axvline(0.5, color="gray", linestyle=":", label="Threshold padrão: 0.50")
axes[0].set_xlabel("Threshold", fontsize=12)
axes[0].set_ylabel("F2-Score", fontsize=12)
axes[0].set_title("F2-Score por Threshold", fontweight="bold")
axes[0].legend()

# Precision vs Recall por threshold
axes[1].plot(thresholds, precision_scores, label="Precision", color="#3498DB", linewidth=2)
axes[1].plot(thresholds, recall_scores,    label="Recall",    color="#2ECC71", linewidth=2)
axes[1].axvline(best_threshold, color="navy", linestyle="--",
                label=f"Threshold ótimo: {best_threshold:.3f}")
axes[1].set_xlabel("Threshold", fontsize=12)
axes[1].set_ylabel("Score", fontsize=12)
axes[1].set_title("Precision e Recall por Threshold", fontweight="bold")
axes[1].legend()

plt.suptitle("Otimização de Threshold", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Relatório com threshold ótimo
y_pred_optimal = (y_proba_best >= best_threshold).astype(int)
print(f"\nClassification Report (threshold={best_threshold:.3f}):")
print(classification_report(y_test, y_pred_optimal))


## 13. Matriz de Confusão e Análise de Erro

In [0]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Threshold padrão
cm_default = confusion_matrix(y_test, (y_proba_best >= 0.5).astype(int))
ConfusionMatrixDisplay(cm_default, display_labels=["Fica","Sai"]).plot(
    ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title(f"Threshold Padrão (0.5)\nF2={fbeta_score(y_test,(y_proba_best>=0.5).astype(int),beta=2):.4f}",
                   fontweight="bold")

# Threshold ótimo
cm_optimal = confusion_matrix(y_test, y_pred_optimal)
ConfusionMatrixDisplay(cm_optimal, display_labels=["Fica","Sai"]).plot(
    ax=axes[1], colorbar=False, cmap="Oranges")
axes[1].set_title(f"Threshold Ótimo ({best_threshold:.3f})\nF2={best_f2_thresh:.4f}",
                   fontweight="bold")

# Custo de erro: FN (não detectado) vs FP (falso alarme)
tn_d, fp_d, fn_d, tp_d = cm_default.ravel()
tn_o, fp_o, fn_o, tp_o = cm_optimal.ravel()

categorias = ["VP (detecções corretas)","FP (falsos alarmes)","FN (saídas não detectadas)","VN"]
valores_default = [tp_d, fp_d, fn_d, tn_d]
valores_opt     = [tp_o, fp_o, fn_o, tn_o]

x = np.arange(4)
width = 0.35
axes[2].bar(x - width/2, valores_default, width, label="Threshold 0.5",  color="#3498DB", alpha=0.8)
axes[2].bar(x + width/2, valores_opt,     width, label=f"Threshold {best_threshold:.2f}", color="#E74C3C", alpha=0.8)
axes[2].set_xticks(x)
axes[2].set_xticklabels(categorias, rotation=15, ha="right", fontsize=9)
axes[2].set_title("Comparação de Erros por Threshold", fontweight="bold")
axes[2].legend()

plt.suptitle("Análise de Confusão — Impacto da Otimização de Threshold", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print(f"\n📊 Redução de Falsos Negativos (saídas não detectadas):")
print(f"   Threshold 0.50 : {fn_d:,} FN")
print(f"   Threshold {best_threshold:.3f}: {fn_o:,} FN  (↓ {fn_d-fn_o:,} casos resgatados)")


## 14. Interpretabilidade — Feature Importance

In [0]:
# Feature importance do LightGBM otimizado (ou melhor pipeline disponível)
try:
    best_pipe = results[best_model_name]["Pipeline"]
    lgbm_clf  = best_pipe.named_steps["classifier"]
    preproc   = best_pipe.named_steps["preprocessor"]

    # Nomes das features após OneHotEncoding
    num_names  = numerical_cols
    cat_names  = list(preproc.named_transformers_["cat"].get_feature_names_out(categorical_cols))
    all_feat_names = num_names + cat_names

    importances = lgbm_clf.feature_importances_
    if len(importances) == len(all_feat_names):
        fi_df = pd.DataFrame({"Feature": all_feat_names, "Importance": importances})
        fi_df = fi_df.sort_values("Importance", ascending=False).head(20)

        fig, ax = plt.subplots(figsize=(10, 8))
        colors_fi = ["#E74C3C" if any(f in feat for f in new_feats) else "#3498DB"
                     for feat in fi_df["Feature"]]
        sns.barplot(x="Importance", y="Feature", data=fi_df, palette=colors_fi, ax=ax)
        ax.set_title(f"Top 20 Features — {best_model_name}\n(vermelho = features criadas)",
                     fontweight="bold")
        plt.tight_layout()
        plt.show()
        print("\n📌 Features criadas destacadas em vermelho — validam o feature engineering.")
    else:
        print("⚠️  Dimensões incompatíveis — usando Random Forest para feature importance.")
        raise ValueError()
except Exception:
    # Fallback: Random Forest
    rf_pipe = results.get("Random Forest", {}).get("Pipeline")
    if rf_pipe:
        rf_clf    = rf_pipe.named_steps["classifier"]
        preproc   = rf_pipe.named_steps["preprocessor"]
        num_names = numerical_cols
        try:
            cat_names = list(preproc.named_transformers_["cat"].get_feature_names_out(categorical_cols))
        except:
            cat_names = []
        all_names = num_names + cat_names
        imps = rf_clf.feature_importances_
        if len(imps) == len(all_names):
            fi_df = pd.DataFrame({"Feature": all_names, "Importance": imps})
            fi_df = fi_df.sort_values("Importance", ascending=False).head(20)
            fig, ax = plt.subplots(figsize=(10, 8))
            sns.barplot(x="Importance", y="Feature", data=fi_df, color="#3498DB", ax=ax)
            ax.set_title("Top 20 Features — Random Forest", fontweight="bold")
            plt.tight_layout()
            plt.show()


## 15. Análise de Viés e Fairness

Verificamos se o modelo apresenta desempenho diferenciado entre grupos demográficos protegidos
(gênero, estado civil), o que poderia caracterizar discriminação algorítmica.


In [0]:
# Reconstruímos o DataFrame de teste com variáveis demográficas para análise de fairness
df_test_fairness = X_test.copy()
df_test_fairness["Attrition_Real"]  = y_test.values
df_test_fairness["Prob_Attrition"]  = y_proba_best
df_test_fairness["Pred_Threshold"]  = y_pred_optimal

print("=" * 65)
print("ANÁLISE DE FAIRNESS — DESEMPENHO POR GRUPO DEMOGRÁFICO")
print("=" * 65)

# Grupos protegidos
grupos = {
    "Gender":       ["Male", "Female"],
    "MaritalStatus": ["Single", "Married", "Divorced"],
}

for grupo_col, categorias in grupos.items():
    print(f"\n📊 Variável: {grupo_col}")
    print(f"  {'Grupo':<15} {'N':>8} {'% Real Attrition':>18} {'% Pred Attrition':>18} {'F2-Score':>10} {'PR-AUC':>10}")
    print("  " + "-" * 80)

    for cat in categorias:
        mask = df_test_fairness[grupo_col] == cat
        sub  = df_test_fairness[mask]
        if len(sub) < 100:
            continue
        n = len(sub)
        real_rate = sub["Attrition_Real"].mean() * 100
        pred_rate = sub["Pred_Threshold"].mean() * 100
        f2_g = fbeta_score(sub["Attrition_Real"], sub["Pred_Threshold"], beta=2, zero_division=0)
        try:
            pr_g = roc_auc_score(sub["Attrition_Real"], sub["Prob_Attrition"])
        except:
            pr_g = float("nan")
        print(f"  {cat:<15} {n:>8,} {real_rate:>17.1f}% {pred_rate:>17.1f}% {f2_g:>10.4f} {pr_g:>10.4f}")

print("\n📌 Interpretação: diferenças de F2-Score entre grupos acima de 0.05 merecem investigação.")
print("   Grupos com menor representação podem ter desempenho preditivo inferior.")


In [0]:
# Visualização de Fairness
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (grupo_col, _) in zip(axes, grupos.items()):
    taxa_real = df_test_fairness.groupby(grupo_col)["Attrition_Real"].mean() * 100
    taxa_pred = df_test_fairness.groupby(grupo_col)["Pred_Threshold"].mean() * 100

    x = np.arange(len(taxa_real))
    width = 0.35
    ax.bar(x - width/2, taxa_real.values, width, label="Real",    color="#3498DB", alpha=0.8)
    ax.bar(x + width/2, taxa_pred.values, width, label="Previsto", color="#E74C3C", alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(taxa_real.index, rotation=15)
    ax.set_ylabel("Taxa de Attrition (%)")
    ax.set_title(f"Fairness: Attrition Real vs Previsto por {grupo_col}", fontweight="bold")
    ax.legend()
    ax.axhline(df_test_fairness["Attrition_Real"].mean()*100,
               color="gray", linestyle="--", label="Média geral")

plt.tight_layout()
plt.show()


## 16. Validação Cruzada Estratificada

In [0]:
# Validação cruzada no conjunto de treino (5-fold) para os dois melhores modelos
cv_models = {
    "LightGBM (balanced)": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LGBMClassifier(n_estimators=200, learning_rate=0.03, max_depth=6,
                                       num_leaves=31, class_weight="balanced",
                                       random_state=42, n_jobs=-1, verbose=-1)),
    ]),
    "XGBoost": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=6,
                                      scale_pos_weight=scale_pos, random_state=42,
                                      n_jobs=-1, eval_metric="logloss", verbosity=0)),
    ]),
}

cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("VALIDAÇÃO CRUZADA ESTRATIFICADA (5-fold) — Métrica: Average Precision (PR-AUC)")
print("=" * 70)

# Usamos amostra para viabilidade computacional em 1M de registros
sample_idx = np.random.RandomState(42).choice(len(X_train),
                                               size=min(100_000, len(X_train)),
                                               replace=False)
X_cv = X_train.iloc[sample_idx]
y_cv = y_train.iloc[sample_idx]

for name, pipe in cv_models.items():
    scores = cross_val_score(pipe, X_cv, y_cv,
                             cv=cv_strat, scoring="average_precision", n_jobs=-1)
    print(f"  {name:<30} PR-AUC: {scores.mean():.4f} ± {scores.std():.4f}  "
          f"[{scores.min():.4f}, {scores.max():.4f}]")

print("\n📌 Nota: CV executado em amostra de 100k registros para viabilidade computacional.")


## 17. Análise de Impacto Financeiro

**Pergunta:** *"Qual o real impacto financeiro de reduzir o attrition em 10%?"*


In [0]:
best_model_name_final = df_metrics.iloc[0]["Modelo"]
y_proba_final = results[best_model_name_final]["y_proba"]

# Premissas do enunciado
custo_total_empresa       = 45_000_000.00    # R$ 45 M de custo anual de attrition
total_desligados_empresa  = int(y_test.sum() / 0.2)  # extrapola teste 20% → 100%
custo_por_saida           = custo_total_empresa / total_desligados_empresa
taxa_retencao_rh          = 0.60             # 60% de sucesso das ações preventivas

# Análise por faixa de risco (decis)
df_rh = X_test.copy()
df_rh["Realidade"]          = y_test.values
df_rh["Probabilidade_Risco"] = y_proba_final
df_rh = df_rh.sort_values("Probabilidade_Risco", ascending=False)

print("=" * 70)
print("ANÁLISE DE IMPACTO FINANCEIRO POR PERCENTIL DE RISCO")
print("=" * 70)
print(f"Custo médio por desligamento: R$ {custo_por_saida:>10,.2f}")
print()

cenarios = [(0.05, "Top 5%"), (0.10, "Top 10%"), (0.20, "Top 20%"), (0.30, "Top 30%")]
for frac, label in cenarios:
    top_n       = int(len(df_rh) * frac)
    assertivos  = df_rh.head(top_n)["Realidade"].sum()
    retidos     = int(assertivos * taxa_retencao_rh)
    economia    = retidos * custo_por_saida * 5   # × 5 para extrapolar ao total da empresa
    precision_g = assertivos / top_n if top_n > 0 else 0
    print(f"  {label:<10} → {top_n:>7,} abordados | "
          f"{assertivos:>7,} verdadeiros ({precision_g:.0%}) | "
          f"retidos: {retidos*5:>7,} | economia: R$ {economia:>12,.0f}")

print()
meta = 45_000_000 * 0.10
print(f"Meta: reduzir 10% do custo → R$ {meta:,.0f}")

# Curva de ROI por percentil
fracs = np.linspace(0.01, 0.50, 100)
economias, precisions = [], []
for frac in fracs:
    top_n = int(len(df_rh) * frac)
    asrt  = df_rh.head(top_n)["Realidade"].sum()
    ret   = int(asrt * taxa_retencao_rh)
    economias.append(ret * custo_por_saida * 5)
    precisions.append(asrt / top_n if top_n > 0 else 0)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(fracs * 100, [e / 1e6 for e in economias],
             color="#2ECC71", linewidth=2.5)
axes[0].axhline(meta / 1e6, color="navy", linestyle="--",
                label=f"Meta: R$ {meta/1e6:.1f}M")
axes[0].set_xlabel("% da Base Abordada pelo RH")
axes[0].set_ylabel("Economia Projetada (R$ Milhões)")
axes[0].set_title("Economia Projetada por Percentual de Abordagem", fontweight="bold")
axes[0].legend()

axes[1].plot(fracs * 100, [p * 100 for p in precisions],
             color="#E74C3C", linewidth=2.5)
axes[1].set_xlabel("% da Base Abordada pelo RH")
axes[1].set_ylabel("Precision (% de verdadeiros positivos)")
axes[1].set_title("Precision por Percentual de Abordagem\n(trade-off: aborda mais, erra mais)", fontweight="bold")

plt.suptitle("Análise de ROI do Modelo Preditivo", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()
